# RAG-chemistry — walkthrough completo del pipeline

Este notebook recorre, paso a paso y con datos reales, todo el pipeline construido en `src/`: extracción de PDFs, chunking, embeddings, índice vectorial con filtros de ruido, retrieval con reranking, generación con citación obligatoria, y evaluación.

**Esto es el detalle técnico interactivo.** Para el *por qué* de cada decisión de arquitectura (y los hallazgos/errores reales encontrados en el camino), ver [`docs/`](../docs/) — cada fase tiene su propio archivo numerado (`01-extraccion.md`, `02-chunking.md`, ... `07-evaluacion.md`).

Requiere que el pipeline ya se haya corrido al menos una vez (`data/processed/` y `vectorstore/` poblados) — este notebook **lee** esos artefactos para mostrarlos, no reconstruye todo desde cero (eso tomaría varios minutos por los modelos de embeddings/reranking).

In [1]:
import sys
from pathlib import Path

# Los notebooks corren con cwd=notebooks/; el resto del pipeline asume cwd=raíz del repo.
if not Path("src").exists():
    sys.path.insert(0, "..")
    import os; os.chdir("..")

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/alexanderosoriohenao/Desktop/RAG-chemistry


## Fase 1 — Extracción

Cada libro en `data/raw/` se convierte en `data/processed/<libro>/pages.jsonl`: un registro JSON por página, con metadata de capítulo ya resuelta (por nombre de archivo o por el TOC embebido del PDF). Ver [`docs/01-extraccion.md`](../docs/01-extraccion.md).

In [2]:
import json

with open("data/processed/coatings_materials_and_surface_coatings/pages.jsonl", encoding="utf-8") as f:
    coatings_pages = [json.loads(line) for line in f]

print(f"Total de páginas extraídas: {len(coatings_pages)}")
print("\nEjemplo de un registro de página:")
sample = coatings_pages[20]
for key, value in sample.items():
    preview = value[:150] + "..." if isinstance(value, str) and len(value) > 150 else value
    print(f"  {key}: {preview!r}")

Total de páginas extraídas: 464

Ejemplo de un registro de página:
  book: 'Coatings Materials and Surface Coatings'
  chapter: '12'
  page: 4
  source_file: '44044ch12.pdf'
  text: '12-4\nCoatings Materials and Surface Coatings\nIt is these exceptional barrier properties that permit the use of extremely thin layers of EVOH resins\nin...'
  extraction_method: 'native'


## Fase 2 — Chunking

Cada capítulo se divide en fragmentos de 600 tokens con 100 de overlap, sin cruzar nunca la frontera de un capítulo. Ver [`docs/02-chunking.md`](../docs/02-chunking.md).

In [3]:
from itertools import groupby
from src.chunking.chunker import chunk_pages

# Capítulo 7 ("The Polyurea Revolution") — el mismo ejemplo usado en docs/02
chapter_7_pages = [p for p in coatings_pages if p["chapter"] == "7"]
chunks_ch7 = chunk_pages(chapter_7_pages)

print(f"Páginas del capítulo 7: {len(chapter_7_pages)} -> {len(chunks_ch7)} chunks")
print(f"\nTokens del chunk 0: {chunks_ch7[0]['n_tokens']}, páginas {chunks_ch7[0]['page_start']}-{chunks_ch7[0]['page_end']}")

# Verificación visual del overlap entre chunks consecutivos
overlap_text = chunks_ch7[0]["text"][-100:]
print(f"\n¿El final del chunk 0 aparece dentro del chunk 1? {overlap_text in chunks_ch7[1]['text']}")

Páginas del capítulo 7: 4 -> 5 chunks

Tokens del chunk 0: 600, páginas 1-2

¿El final del chunk 0 aparece dentro del chunk 1? True


## Fase 3 — Embeddings

Cada chunk se vectoriza con un modelo multilingüe (`intfloat/multilingual-e5-base`) para que una pregunta en español pueda recuperar contenido en inglés. Ver [`docs/03-embeddings.md`](../docs/03-embeddings.md).

*(Esta celda carga el modelo de embeddings — puede tardar uno o dos minutos la primera vez que corre.)*

In [4]:
from src.embeddings.embedder import embed_query
import numpy as np

with open("data/processed/coatings_materials_and_surface_coatings/chunks.jsonl", encoding="utf-8") as f:
    coatings_chunks = [json.loads(line) for line in f]
coatings_vectors = np.load("data/processed/coatings_materials_and_surface_coatings/embeddings.npy")
print(f"Vectores de Coatings Materials: {coatings_vectors.shape}")

q_vec = embed_query("¿Qué es el poliurea?")
sims = coatings_vectors @ q_vec
best = int(np.argmax(sims))
best_chunk = coatings_chunks[best]
print(f"\nChunk más similar (similitud={sims[best]:.3f}): capítulo {best_chunk['chapter']}, p.{best_chunk['page_start']}-{best_chunk['page_end']}")

Vectores de Coatings Materials: (686, 768)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Chunk más similar (similitud=0.771): capítulo 7, p.3-4


## Fase 4 — Índice vectorial y filtros de ruido

FAISS combina los chunks de todos los libros en un solo índice. Dos filtros, aplicados solo al construir el índice (nunca a los datos), excluyen secciones de bajo valor: bibliografía (heurística de regex) y secciones estructurales como `Index`/`Table of Contents` (coincidencia exacta de metadata). Ver [`docs/04-indexacion.md`](../docs/04-indexacion.md) y [`docs/07-evaluacion.md`](../docs/07-evaluacion.md) (el filtro estructural se agregó durante la evaluación, no en esta fase original).

In [5]:
from src.indexing.reference_filter import is_bibliography_like
from src.indexing.structural_filter import is_structural_section

bibliography_example = chunks_ch7[-1]["text"]  # el último chunk del capítulo suele tocar las referencias
print("¿Este chunk se detecta como bibliografía?", is_bibliography_like(bibliography_example))
print("¿'Index' se detecta como sección estructural?", is_structural_section("Index"))
print("¿'5. Corrosion Failures' se detecta como estructural? (no debería)", is_structural_section("5. Corrosion Failures"))

¿Este chunk se detecta como bibliografía? False
¿'Index' se detecta como sección estructural? True
¿'5. Corrosion Failures' se detecta como estructural? (no debería) False


## Fase 5 — Retrieval: FAISS + reranking

Comparación directa, con la misma query usada en todas las fases anteriores: búsqueda FAISS pura vs. el resultado final tras el reranking con cross-encoder. Ver [`docs/05-retrieval.md`](../docs/05-retrieval.md).

*(Carga el modelo de reranking — otro minuto la primera vez.)*

In [6]:
from src.indexing.search import search as faiss_search
from src.retrieval.retrieve import retrieve

query = "¿Qué es la corrosión por picadura?"

print("=== Top-3 solo FAISS (sin reranking) ===")
for r in faiss_search(query, k=3):
    print(f"  score={r['score']:.3f} [{r['book']}, cap. {r['chapter']}, p.{r['page_start']}-{r['page_end']}]")

print("\n=== Top-3 tras reranking con cross-encoder ===")
for r in retrieve(query, top_k=3):
    print(f"  rerank_score={r['rerank_score']:.4f} [{r['book']}, cap. {r['chapter']}, p.{r['page_start']}-{r['page_end']}]")

=== Top-3 solo FAISS (sin reranking) ===
  score=0.803 [Handbook of Corrosion Engineering, cap. 5. Corrosion Failures, p.379-379]
  score=0.799 [Handbook of Corrosion Engineering, cap. Introduction, p.21-21]
  score=0.799 [Handbook of Corrosion Engineering, cap. 7. Acceleration and Amplification of Corrosion Damage, p.584-584]

=== Top-3 tras reranking con cross-encoder ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  rerank_score=0.0005 [Handbook of Corrosion Engineering, cap. Corrosion Maintenance through Inspection and Monitoring, p.428-430]
  rerank_score=0.0002 [Handbook of Corrosion Engineering, cap. 5. Corrosion Failures, p.345-346]
  rerank_score=0.0001 [Handbook of Corrosion Engineering, cap. 5. Corrosion Failures, p.343-345]


## Fase 6 — Generación con citación obligatoria

El sistema completo: retrieval + prompt con grounding + generación local con Ollama (`qwen2.5:7b-instruct`). Tres casos: pregunta en español, pregunta en inglés, y una pregunta fuera de dominio (donde lo correcto es que el modelo se niegue a responder). Ver [`docs/06-generacion.md`](../docs/06-generacion.md).

*(Cada llamada genera con el LLM local — puede tardar entre 10 y 40 segundos según la carga de memoria de la máquina.)*

In [7]:
from src.generation.answer import answer_question

for q in [
    "¿Qué es la corrosión por picadura?",
    "What is polyurea used for in protective coatings?",
    "¿Cuál es la capital de Mongolia?",
]:
    result = answer_question(q)
    print(f"PREGUNTA: {q}")
    print(f"RESPUESTA: {result['answer']}")
    print("-" * 80)

PREGUNTA: ¿Qué es la corrosión por picadura?
RESPUESTA: La corrosión por picadura es un tipo de corrosión localizada que se caracteriza por la formación de cavidades o "huecos" en el material [Handbook of Corrosion Engineering, 5. Corrosion Failures, p. 343-345]. Estas cavidades pueden tener diferentes formas, como boca abierta (no cubierta) o tapadas por una membrana semipermeable de productos de corrosión [Handbook of Corrosion Engineering, 5. Corrosion Failures, p. 345-346]. La corrosión por picadura puede ser más peligrosa que la corrosión uniforme debido a su dificultad para detectarse, predecirse y diseñar contra ella [Handbook of Corrosion Engineering, 5. Corrosion Failures, p. 343-345].
--------------------------------------------------------------------------------


PREGUNTA: What is polyurea used for in protective coatings?
RESPUESTA: Polyurea is used for creating protective coatings due to its rapid curing ability, adhesion qualities, and resistance to physical rupture, water, chemicals, and weather [Book, Chapter 7, p. 2-3]. It can be applied as a single coat on various surfaces including vertical and overhead areas, offering minimal downtime and strong sealing characteristics [Book, Chapter 7, p. 3-4].
--------------------------------------------------------------------------------


PREGUNTA: ¿Cuál es la capital de Mongolia?
RESPUESTA: El tema de la pregunta sobre la capital de Mongolia no se encuentra en losexcerptos proporcionados, que se centran en corrupción térmica, tratamientos superficiales y selección de materiales. No hay información relevante para responder a esta pregunta aquí.
--------------------------------------------------------------------------------


## Fase 7 — Evaluación

15 preguntas reales (verificadas contra el corpus), puntuadas con métricas deterministas — sin LLM-as-judge. La evaluación encontró y corrigió 3 problemas reales durante esta fase (detalle completo en [`docs/07-evaluacion.md`](../docs/07-evaluacion.md)): páginas de índice citadas en rechazos, el modelo citando fuentes incluso al negarse a responder, y un caso de prueba mal diseñado.

Esta celda solo carga el resultado de la última corrida guardada — para volver a correrla: `python -m src.evaluation.run_eval`.

In [8]:
results_dir = Path("eval/results")
latest = sorted(results_dir.glob("*.json"))[-1]

with open(latest, encoding="utf-8") as f:
    eval_data = json.load(f)

print(f"Resultados de: {latest.name}\n")
for key, value in eval_data["summary"].items():
    print(f"  {key}: {value}")

Resultados de: 20260625_101610.json

  n_in_domain: 10
  n_out_of_domain: 5
  keyword_hit_rate: 1.0
  book_hit_rate: 1.0
  citation_validity_rate: 0.8
  out_of_domain_refusal_rate: 1.0
  out_of_domain_fabricated_citation_rate: 0.2
  avg_latency_seconds: 12.4


## Cómo reproducir el pipeline completo desde cero

```bash
python -m src.extraction.run      # data/raw/  -> data/processed/*/pages.jsonl
python -m src.chunking.run        # pages.jsonl -> chunks.jsonl
python -m src.embeddings.run      # chunks.jsonl -> embeddings.npy
python -m src.indexing.build_index  # -> vectorstore/index.faiss
python -m src.evaluation.run_eval   # corre las 15 preguntas de prueba
```

Repositorio: [github.com/Halexoh/RAG-chemistry](https://github.com/Halexoh/RAG-chemistry)